# PFM-xLSTM-MIL for WSI Classification

This notebook implements a complete **mock computational pathology pipeline** for bag-level WSI classification.
It is structured to run linearly from top to bottom on a high-memory GPU.

## Cell 1 - Environment Setup and Library Imports

Install required dependencies and configure runtime (device, seeds, imports).

In [ ]:
!pip -q install timm h5py scikit-learn matplotlib hilbertsfc
!pip -q install git+https://github.com/NX-AI/xlstm
!pip -q install git+https://github.com/NX-AI/vision-lstm

import os
import math
import time
import random
import warnings
from dataclasses import dataclass
from pathlib import Path

import h5py
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset

import timm

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Device setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'CUDA capability: {torch.cuda.get_device_capability(0)}')
else:
    warnings.warn('CUDA is not available. This notebook is designed for a high-memory GPU.')


## Cell 2 - Dataset Preparation and Preprocessing Mockup

Create a robust dummy dataset of **50 bags**, each with **10,000 patches** and retained physical `(x, y)` coordinates on a `100x100` grid.

To avoid massive RAM usage, patch tensors are generated lazily in micro-batches via a deterministic function.

In [ ]:
@dataclass
class BagSpec:
    bag_id: int
    label: int
    n_patches: int
    grid_size: int
    seed: int


class DummyWSIBagDataset(Dataset):
    """Mock WSI bag dataset with deterministic lazy patch generation.

    Critical constraints from the task:
    - 50 total bags
    - 10,000 patches per bag
    - patch shape = (3, 224, 224)
    - retain physical (x, y) coordinates relative to a 100x100 grid
    """
    def __init__(self, n_bags=50, n_patches=10_000, grid_size=100):
        assert grid_size * grid_size == n_patches, 'grid_size^2 must equal n_patches for this mock.'
        self.n_bags = n_bags
        self.n_patches = n_patches
        self.grid_size = grid_size

        # Construct deterministic binary labels (balanced).
        labels = np.array([0] * (n_bags // 2) + [1] * (n_bags - n_bags // 2), dtype=np.int64)
        rng = np.random.default_rng(SEED)
        rng.shuffle(labels)

        self.bags = [
            BagSpec(
                bag_id=i,
                label=int(labels[i]),
                n_patches=n_patches,
                grid_size=grid_size,
                seed=SEED + i * 17,
            )
            for i in range(n_bags)
        ]

        # Fixed 2D coordinate grid for each bag (100x100 -> 10,000 tokens).
        xs, ys = np.meshgrid(np.arange(grid_size), np.arange(grid_size), indexing='xy')
        self.coords = np.stack([xs.reshape(-1), ys.reshape(-1)], axis=1).astype(np.int32)

    def __len__(self):
        return len(self.bags)

    def __getitem__(self, idx):
        bag = self.bags[idx]
        return {
            'bag_id': bag.bag_id,
            'label': bag.label,
            'coords': self.coords.copy(),
            'n_patches': bag.n_patches,
            'seed': bag.seed,
        }

    @staticmethod
    def generate_patch_batch(bag_seed, start_idx, end_idx, patch_shape=(3, 224, 224)):
        """Generate deterministic dummy images for a contiguous patch index range.

        Returns a tensor of shape [B, 3, 224, 224] in float32.
        """
        batch_size = end_idx - start_idx
        g = torch.Generator(device='cpu')
        g.manual_seed(int(bag_seed + start_idx * 13))
        # Random image-like values in [0, 1].
        x = torch.rand((batch_size, *patch_shape), generator=g, dtype=torch.float32)
        return x


dataset = DummyWSIBagDataset(n_bags=50, n_patches=10_000, grid_size=100)
print('Number of bags:', len(dataset))
example = dataset[0]
print('Bag example keys:', list(example.keys()))
print('Coordinates shape:', example['coords'].shape)
print('Label:', example['label'])


## Cell 3 - Feature Extraction via Pathology Foundation Model (PFM)

Instantiate a frozen pretrained model from `timm`, extract per-patch embeddings in `torch.no_grad()` mode, and save each bag to an HDF5 file.

We use micro-batching to control GPU memory while processing 10,000 patches per bag.

In [ ]:
H5_PATH = Path('mock_wsi_features.h5')
PFM_MODEL_NAME = 'vit_tiny_patch16_224'
TARGET_EMBED_DIM = 768
EXTRACTION_MICROBATCH = 64

# Build a timm backbone. num_classes=0 returns pooled embeddings.
pfm = timm.create_model(PFM_MODEL_NAME, pretrained=True, num_classes=0)
pfm.eval().to(device)
for p in pfm.parameters():
    p.requires_grad = False

with torch.no_grad():
    test_out = pfm(torch.randn(2, 3, 224, 224, device=device))
in_dim = int(test_out.shape[-1])
print(f'PFM model: {PFM_MODEL_NAME}, output dim: {in_dim}')

# If model output dim is not 768, we project embeddings to 768 before saving.
feature_projector = None
if in_dim != TARGET_EMBED_DIM:
    feature_projector = nn.Linear(in_dim, TARGET_EMBED_DIM, bias=False).to(device)
    feature_projector.eval()
    for p in feature_projector.parameters():
        p.requires_grad = False

def extract_bag_features(bag_record):
    n_patches = bag_record['n_patches']
    bag_seed = bag_record['seed']
    feats = []
    with torch.no_grad():
        for start in range(0, n_patches, EXTRACTION_MICROBATCH):
            end = min(start + EXTRACTION_MICROBATCH, n_patches)
            x = DummyWSIBagDataset.generate_patch_batch(bag_seed, start, end).to(device, non_blocking=True)
            z = pfm(x)
            if feature_projector is not None:
                z = feature_projector(z)
            feats.append(z.detach().cpu().numpy().astype(np.float16))
    return np.concatenate(feats, axis=0)

if H5_PATH.exists():
    H5_PATH.unlink()

t0 = time.time()
with h5py.File(H5_PATH, 'w') as h5:
    for i in range(len(dataset)):
        bag = dataset[i]
        bag_feats = extract_bag_features(bag)
        grp = h5.create_group(f'bag_{bag["bag_id"]:03d}')
        grp.create_dataset('features', data=bag_feats, compression='gzip')
        grp.create_dataset('coords', data=bag['coords'], compression='gzip')
        grp.create_dataset('label', data=np.array([bag['label']], dtype=np.int64))
        if (i + 1) % 5 == 0:
            print(f'Saved {i+1}/{len(dataset)} bags...')

print(f'Feature extraction complete. H5 saved at: {H5_PATH.resolve()}')
print(f'Total extraction time: {(time.time() - t0):.2f} sec')


## Cell 4 - 2D Topology Preservation via Hilbert Curve Sorting

Load features and coordinates from HDF5, compute Hilbert index for each `(x, y)`, and sort the token sequence accordingly.

**Why this matters:** Hilbert ordering preserves local spatial neighborhoods from 2D tissue layout when converted into a 1D sequence. This lets sequence models consume WSI tokens while keeping adjacent tissue patches near each other in sequence order, without convolutional operators.

In [ ]:
def hilbert_indices_from_coords(coords_np, grid_size=100):
    """Compute Hilbert index for each (x, y) coordinate.

    We try the installed hilbertsfc API first. If API differs by version,
    we fall back to a stable deterministic ordering so the notebook remains runnable.
    """
    x = coords_np[:, 0].astype(np.int64)
    y = coords_np[:, 1].astype(np.int64)

    try:
        # Common API style in hilbertsfc versions.
        import hilbertsfc
        if hasattr(hilbertsfc, 'hilbert_index'):
            return np.array([hilbertsfc.hilbert_index(int(a), int(b), p=int(math.log2(grid_size))) for a, b in zip(x, y)])
        if hasattr(hilbertsfc, 'encode'):
            return np.array(hilbertsfc.encode(np.stack([x, y], axis=1)))
    except Exception as e:
        warnings.warn(f'hilbertsfc API fallback triggered: {e}')

    # Deterministic fallback (not true Hilbert) only if runtime API mismatch occurs.
    warnings.warn('Using fallback lexicographic ordering due to hilbertsfc API mismatch.')
    return y * grid_size + x


class SortedH5WSIDataset(Dataset):
    def __init__(self, h5_path):
        self.h5_path = str(h5_path)
        with h5py.File(self.h5_path, 'r') as h5:
            self.keys = sorted(list(h5.keys()))

    def __len__(self):
        return len(self.keys)

    def __getitem__(self, idx):
        key = self.keys[idx]
        with h5py.File(self.h5_path, 'r') as h5:
            grp = h5[key]
            features = grp['features'][:].astype(np.float32)
            coords = grp['coords'][:].astype(np.int32)
            label = int(grp['label'][0])

        hidx = hilbert_indices_from_coords(coords, grid_size=100)
        order = np.argsort(hidx)

        features = torch.tensor(features[order], dtype=torch.float32)
        coords = torch.tensor(coords[order], dtype=torch.int32)
        label = torch.tensor([label], dtype=torch.float32)
        return {'features': features, 'coords': coords, 'label': label, 'bag_key': key}


sorted_dataset = SortedH5WSIDataset(H5_PATH)
sample = sorted_dataset[0]
print('Sorted feature shape:', tuple(sample['features'].shape))
print('Sorted coord shape:', tuple(sample['coords'].shape))
print('Label shape:', tuple(sample['label'].shape))


## Cell 5 - PFM-xLSTM-MIL Aggregator

Define an MIL aggregator with:
- input projection from PFM feature dimension to hidden dimension,
- stacked `mLSTMBlock` layers (alternating sequence direction for bidirectional context),
- bilateral token pooling (`first || last`) and MLP classification head.

In [ ]:
# Try importing mLSTMBlock from vision_lstm; provide robust compatibility fallback.
mLSTMBlock = None
try:
    from vision_lstm.vision_lstm2 import mLSTMBlock  # one possible path
except Exception:
    try:
        from vision_lstm.modules.xlstm import mLSTMBlock  # alternate path
    except Exception:
        warnings.warn('Could not import vision_lstm mLSTMBlock; using nn.LSTM compatibility block.')

if mLSTMBlock is None:
    class mLSTMBlock(nn.Module):
        def __init__(self, dim):
            super().__init__()
            self.lstm = nn.LSTM(input_size=dim, hidden_size=dim, num_layers=1, batch_first=True)
            self.norm = nn.LayerNorm(dim)
        def forward(self, x):
            y, _ = self.lstm(x)
            return self.norm(y + x)


class PFMxLSTMMIL(nn.Module):
    def __init__(self, in_dim=768, hidden_dim=512, num_blocks=2, mlp_dim=256, dropout=0.2):
        super().__init__()
        assert num_blocks >= 2, 'Use at least two mLSTM blocks for alternating direction.'
        self.input_proj = nn.Linear(in_dim, hidden_dim)
        self.blocks = nn.ModuleList([mLSTMBlock(hidden_dim) for _ in range(num_blocks)])

        self.cls_head = nn.Sequential(
            nn.Linear(hidden_dim * 2, mlp_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_dim, 1),
        )

    def forward(self, x):
        # x shape: [B, N, in_dim]
        x = self.input_proj(x)

        # Odd-numbered blocks: forward order.
        # Even-numbered blocks: reverse order using flip before and after block.
        for i, block in enumerate(self.blocks):
            block_id = i + 1
            if block_id % 2 == 1:
                x = block(x)
            else:
                xr = torch.flip(x, dims=[1])
                xr = block(xr)
                x = torch.flip(xr, dims=[1])

        # Bilateral concatenation pooling: first token + last token.
        first_tok = x[:, 0, :]
        last_tok = x[:, -1, :]
        pooled = torch.cat([first_tok, last_tok], dim=-1)
        logit = self.cls_head(pooled)
        return logit


model = PFMxLSTMMIL(in_dim=768, hidden_dim=512, num_blocks=2, mlp_dim=256).to(device)
print(model.__class__.__name__)


## Cell 6 - Training and Validation Loop

Train for 10 epochs with batch size = 1 bag (10,000 tokens), track losses, save best model state, and record peak CUDA memory each epoch.

In [ ]:
# Build train/val split over bag indices
all_indices = np.arange(len(sorted_dataset))
all_labels = np.array([int(sorted_dataset[i]['label'].item()) for i in all_indices])
train_idx, val_idx = train_test_split(
    all_indices,
    test_size=0.2,
    random_state=SEED,
    stratify=all_labels,
)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

EPOCHS = 10
train_losses = []
val_losses = []
gpu_peak_mem_gb = []
best_val_loss = float('inf')
best_state = None

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_train_loss = 0.0

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats(device)

    for idx in train_idx:
        bag = sorted_dataset[int(idx)]
        x = bag['features'].unsqueeze(0).to(device)   # [1, 10000, 768]
        y = bag['label'].to(device)                   # [1]

        optimizer.zero_grad(set_to_none=True)
        logit = model(x).squeeze(-1)
        loss = criterion(logit, y)
        loss.backward()
        optimizer.step()

        epoch_train_loss += float(loss.item())

    epoch_train_loss /= len(train_idx)
    train_losses.append(epoch_train_loss)

    model.eval()
    epoch_val_loss = 0.0
    with torch.no_grad():
        for idx in val_idx:
            bag = sorted_dataset[int(idx)]
            x = bag['features'].unsqueeze(0).to(device)
            y = bag['label'].to(device)
            logit = model(x).squeeze(-1)
            loss = criterion(logit, y)
            epoch_val_loss += float(loss.item())

    epoch_val_loss /= len(val_idx)
    val_losses.append(epoch_val_loss)

    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if torch.cuda.is_available():
        peak_mem = torch.cuda.max_memory_allocated(device) / (1024 ** 3)
    else:
        peak_mem = 0.0
    gpu_peak_mem_gb.append(float(peak_mem))

    print(
        f'Epoch {epoch:02d}/{EPOCHS} | '
        f'Train Loss: {epoch_train_loss:.4f} | '
        f'Val Loss: {epoch_val_loss:.4f} | '
        f'Peak GPU Mem (GB): {peak_mem:.3f}'
    )

if best_state is not None:
    model.load_state_dict(best_state)
    print('Loaded best model weights based on validation loss.')


## Cell 7 - Evaluation, Metrics, and Visualization

Compute final classification metrics and create three requested plots:
1. ROC curve
2. GPU memory scaling across epochs
3. Training/validation loss curves

In [ ]:
def evaluate_model(model, dataset_obj, indices, device):
    model.eval()
    y_true = []
    y_prob = []
    with torch.no_grad():
        for idx in indices:
            bag = dataset_obj[int(idx)]
            x = bag['features'].unsqueeze(0).to(device)
            y = float(bag['label'].item())
            logit = model(x).squeeze().item()
            prob = 1.0 / (1.0 + math.exp(-logit))
            y_true.append(y)
            y_prob.append(prob)

    y_true = np.array(y_true, dtype=np.int64)
    y_prob = np.array(y_prob, dtype=np.float64)
    y_pred = (y_prob >= 0.5).astype(np.int64)

    metrics = {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'roc_auc': roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else float('nan'),
    }
    return metrics, y_true, y_prob


metrics, y_true, y_prob = evaluate_model(model, sorted_dataset, val_idx, device)
print('Final Validation Metrics')
for k, v in metrics.items():
    print(f'  {k}: {v:.4f}')

# Plot 1: ROC Curve
plt.figure(figsize=(6, 5))
if len(np.unique(y_true)) > 1:
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc = roc_auc_score(y_true, y_prob)
    plt.plot(fpr, tpr, label=f'ROC AUC = {auc:.3f}', linewidth=2)
else:
    plt.plot([0, 1], [0, 1], '--', label='Insufficient class variety for ROC')
plt.plot([0, 1], [0, 1], 'k--', alpha=0.6)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve (Validation Set)')
plt.legend()
plt.grid(alpha=0.2)
plt.show()

# Plot 2: GPU Memory Scaling
plt.figure(figsize=(7, 4))
epochs_axis = np.arange(1, len(gpu_peak_mem_gb) + 1)
plt.plot(epochs_axis, gpu_peak_mem_gb, marker='o', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Peak GPU Memory (GB)')
plt.title('GPU Memory Scaling During Training')
plt.grid(alpha=0.2)
plt.show()

# Plot 3: Loss Curves
plt.figure(figsize=(7, 4))
plt.plot(epochs_axis, train_losses, marker='o', label='Train Loss')
plt.plot(epochs_axis, val_losses, marker='s', label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss Curves')
plt.legend()
plt.grid(alpha=0.2)
plt.show()
